In [1]:
import pandas as pd
import numpy as np
from sklearn.ensemble import IsolationForest
import joblib
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import StandardScaler

In [2]:
data = pd.read_csv("/kaggle/input/datasets/itsmonir31/hydroponics-datasets/IoTData --Raw--.csv")

In [3]:
data.columns

Index(['id', 'timestamp', 'pH', 'TDS', 'water_level', 'DHT_temp',
       'DHT_humidity', 'water_temp', 'pH_reducer', 'add_water',
       'nutrients_adder', 'humidifier', 'ex_fan'],
      dtype='object')

In [4]:
actuators = ["pH_reducer","add_water","nutrients_adder","humidifier","ex_fan"]

for col in actuators:
    data[col] = data[col].astype(str).str.strip().str.upper()

In [5]:
for col in actuators:
    data[col] = data[col].map({"OFF":0,"ON":1})

In [6]:
data

,id,timestamp,pH,TDS,water_level,DHT_temp,DHT_humidity,water_temp,pH_reducer,add_water,nutrients_adder,humidifier,ex_fan
0,1,2023-11-26 10:57:52,7.00,500.00,0,25.5,60.0,20.00,1,NaN,0,0,1
1,2,2023-11-26 10:58:37,7.00,500.00,0,25.5,60.0,20.00,1,NaN,0,0,0
2,3,2023-11-26 11:01:34,7.00,500.00,3,25.5,60.0,20.00,1,NaN,0,0,0
3,4,2023-12-05 11:30:58,7.00,500.00,3,25.5,60.0,20.00,1,NaN,0,0,0
4,5,2023-12-05 11:33:50,7.00,500.00,3,25.5,60.0,20.00,1,1.0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
50565,50566,2023-12-26 21:34:40,5.54,1125.10,1,23.9,77.7,21.82,0,0.0,0,0,0
50566,50567,2023-12-26 21:34:50,5.55,1125.10,1,23.9,77.7,19.83,0,0.0,0,0,0
50567,50568,2023-12-26 21:35:00,5.57,1125.21,1,23.9,77.7,19.41,0,0.0,0,0,0
50568,50569,2023-12-26 21:35:10,5.56,1124.89,1,23.9,77.7,19.88,0,0.0,0,0,0


In [7]:
print(data.isna().sum())

id                 0
timestamp          0
pH                 0
TDS                0
water_level        0
DHT_temp           0
DHT_humidity       0
water_temp         0
pH_reducer         0
add_water          4
nutrients_adder    0
humidifier         0
ex_fan             0
dtype: int64


In [8]:
data = data.dropna()

In [30]:
# label water change
data['change_needed'] = ((data['pH'] < 5.5) | (data['pH'] > 6.8) |
                       (data['TDS'] < 500) | (data['TDS'] > 2000) |
                       (data['water_level'] == 0)).astype(int)
data

,id,timestamp,pH,TDS,water_level,DHT_temp,DHT_humidity,water_temp,pH_reducer,add_water,nutrients_adder,humidifier,ex_fan,change_needed
4,5,2023-12-05 11:33:50,7.00,500.00,3,25.5,60.0,20.00,1,1.0,0,0,0,1
5,6,2023-12-05 11:34:58,7.00,500.00,3,25.5,60.0,20.00,1,1.0,0,0,0,1
6,7,2023-12-05 11:42:40,5.67,500.00,3,25.5,60.0,20.00,1,1.0,0,0,0,0
7,8,2023-12-05 11:42:50,5.71,500.00,3,25.5,60.0,20.00,1,1.0,0,0,0,0
8,9,2023-12-05 11:43:00,5.66,500.00,3,25.5,60.0,20.00,1,1.0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
50565,50566,2023-12-26 21:34:40,5.54,1125.10,1,23.9,77.7,21.82,0,0.0,0,0,0,0
50566,50567,2023-12-26 21:34:50,5.55,1125.10,1,23.9,77.7,19.83,0,0.0,0,0,0,0
50567,50568,2023-12-26 21:35:00,5.57,1125.21,1,23.9,77.7,19.41,0,0.0,0,0,0,0
50568,50569,2023-12-26 21:35:10,5.56,1124.89,1,23.9,77.7,19.88,0,0.0,0,0,0,0


In [31]:
print("Original:", data.shape)

data = data[data["pH"].between(4,9)]
print("After pH filter:", data.shape)

data = data[data["TDS"] > 0]
print("After TDS filter:", data.shape)

data = data[data["DHT_humidity"] < 100]
print("After humidity filter:", data.shape)

Original: (50314, 14)
After pH filter: (50314, 14)
After TDS filter: (50314, 14)
After humidity filter: (50314, 14)


In [32]:
X = data[['pH', 'TDS', 'water_level', 'DHT_temp','DHT_humidity', 'water_temp']]
y = data[['change_needed']]

In [33]:
X = X.values
y = y.values

In [34]:
X_train, X_test, y_train, y_test = train_test_split(
X, y, test_size=0.2, random_state=42
)

In [14]:
pip install mindspore

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 766.5/766.5 MB 2.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 71.6 MB/s eta 0:00:00:00:0100:01
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully uninstalled numpy-2.0.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.31.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
cesium 0.12.4 requires numpy<3.0,>=2.0, but you have numpy 1.26.4 which is incompatible.
kaggle-environments 1.27.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 1.0.0 requi

In [35]:
from mindspore import Tensor
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

X_train = ms.Tensor(X_train, ms.float32)
X_test = ms.Tensor(X_test, ms.float32)

In [36]:
X_train = Tensor(X_train.astype(np.float32))
X_test = Tensor(X_test.astype(np.float32))

y_train = Tensor(y_train.astype(np.float32))
y_test = Tensor(y_test.astype(np.float32))

In [37]:
print(X_train.dtype)
print(y_train.dtype)

Float32
Float32


In [39]:
import mindspore as ms
from mindspore import nn



class HydroponicModel(nn.Cell):
    def __init__(self):
        super().__init__()
        self.net = nn.SequentialCell(
            nn.Dense(X_train.shape[1], 64),
            nn.ReLU(),
            nn.Dense(64, 32),
            nn.ReLU(),
            nn.Dense(32, 16),
            nn.ReLU(),
            nn.Dense(16, 1)
        )

    def construct(self, x):
        return self.net(x)

model = HydroponicModel()

# loss function for multi-label classification
loss_fn = nn.BCEWithLogitsLoss()


optimizer = nn.Adam(model.trainable_params(), learning_rate=0.001)

train_net = nn.TrainOneStepCell(nn.WithLossCell(model, loss_fn), optimizer)

for epoch in range(1000):
    loss = train_net(X_train, y_train)
    if epoch % 50 == 0 :
        print("Epoch:", epoch, "Loss:", loss)

Epoch: 0 Loss: 0.66299534
Epoch: 50 Loss: 0.397471
Epoch: 100 Loss: 0.24962288
Epoch: 150 Loss: 0.13042359
Epoch: 200 Loss: 0.07789087
Epoch: 250 Loss: 0.046794288
Epoch: 300 Loss: 0.03348935
Epoch: 350 Loss: 0.025621988
Epoch: 400 Loss: 0.020384746
Epoch: 450 Loss: 0.016701892
Epoch: 500 Loss: 0.014088611
Epoch: 550 Loss: 0.011289512
Epoch: 600 Loss: 0.009667882
Epoch: 650 Loss: 0.008498481
Epoch: 700 Loss: 0.007575536
Epoch: 750 Loss: 0.0068309614
Epoch: 800 Loss: 0.0062015927
Epoch: 850 Loss: 0.0056872624
Epoch: 900 Loss: 0.0052424893
Epoch: 950 Loss: 0.004859678


In [40]:
import mindspore.ops as ops

pred = ops.sigmoid(model(X_test))

print(pred[:10])

[[6.2840327e-06]
 [7.5890033e-10]
 [7.6616755e-11]
 [1.1378709e-06]
 [6.0060522e-08]
 [5.7875747e-07]
 [2.1101336e-05]
 [1.0000000e+00]
 [2.3519249e-06]
 [6.3857610e-06]]


In [43]:
pred = ops.sigmoid(model(X_test))

pred_binary = (pred.asnumpy() > 0.1).astype(int)

print(pred_binary[:10])
print(y_test[:10])

[[0]
 [0]
 [0]
 [0]
 [0]
 [0]
 [0]
 [1]
 [0]
 [0]]
[[0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [1.]
 [0.]
 [0.]]


In [44]:
import mindspore as ms

ms.save_checkpoint(model, "mindspore_hydroponic_model.ckpt")

In [ ]:
model = HydroponicModel()

param_dict = ms.load_checkpoint("hydroponic_model.ckpt")
ms.load_param_into_net(model, param_dict)

In [ ]:
import mindspore.ops as ops

sigmoid = ops.Sigmoid()

logits = model(X_test)
probs = sigmoid(logits)

y_pred = (probs > 0.5).astype(ms.int32)

print(y_pred)